In [ ]:
%cd ../..

In [ ]:
from scripts.utils import *
from scripts.globals import *
import editdistance

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv(f"{XGB_PIPELINE_DIR}/1_find_matches_for_clash.csv")
clash_df = pd.read_csv(CLASH_CSV)
df.head()

In [ ]:
negative_df = df[["mre_region", "mirna_sequence", "mirna_accession"]]

# shuffle mirna sequences using levenstein distance
for i, row in negative_df.iterrows():
    original_string = row['mirna_sequence']
    # exclude the original string from the column
    column = negative_df['mirna_sequence'].drop(i)
    new_string = find_most_different_string(original_string, column)
    negative_df.at[i, 'shuffled_sequence'] = new_string

# renaming cols for convention
rename_dict = {"mirna_sequence": "old_mirna_sequence",
               "mirna_accession": "old_mirna_accession", "shuffled_sequence": "mirna_sequence"}
negative_df = negative_df.rename(columns=rename_dict)

# adding accessions for new mirna sequences
accession_dict = clash_df.set_index('mirna_sequence')[
    'mirna_accession'].to_dict()
negative_df['mirna_accession'] = negative_df['mirna_sequence'].map(
    accession_dict)

In [ ]:
# renaming col to be able to run find_matches_with_rnaduplex()
negative_df = negative_df.rename(columns={"mre_region": "mrna_sequence"})

# finding matches on negative data
negative_df_results = find_matches_with_rnaduplex(negative_df)

In [ ]:
# adding relevant columns
negative_df_results["mirna_sequence"] = negative_df["mirna_sequence"]
negative_df_results["mre_region"] = negative_df["mrna_sequence"]
negative_df_results["mirna_accession"] = negative_df["mirna_accession"]

# adding columns from CLASH
negative_df_results["id"] = clash_df["id"]
negative_df_results["enst"] = clash_df["enst"]
negative_df_results["extended_mrna_start"] = clash_df["extended_start"]
negative_df_results["extended_mrna_end"] = clash_df["extended_end"]
negative_df_results["extended_mrna_sequence"] = clash_df["extended_sequence"]
negative_df_results["clash_mrna_start"] = clash_df["true_start_index"]
negative_df_results["clash_mrna_end"] = clash_df["true_end_index"]
negative_df_results["full_sequence_of_transcript"] = clash_df["full_sequence_of_transcript"]

negative_df_results["mre_start"] = df["mre_start"]
negative_df_results["mre_end"] = df["mre_end"]

In [ ]:
# export to csv
negative_df_results.to_csv(
    f"{XGB_PIPELINE_DIR}/3_negative_data.csv", index=False)